# 04 Validation

Check Monte Carlo convergence, simple probability scoring, and an **empirical vs Poisson** plot of *daily* crash counts for one ZIP / day / hour bucket.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.validation import brier_score, convergence_test, simple_train_test_split_by_date
from src.visualization import plot_empirical_poisson_check

In [ ]:
rate_table = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "crash_rate_table.csv", dtype={"zip_code": str})
row = rate_table.iloc[0]

convergence_df = convergence_test(
    crash_count=int(row["crash_count"]),
    observed_hours=int(row["observed_hours"]),
    weather_condition="clear",
    random_seed=42,
)
convergence_df

In [ ]:
example_probability = convergence_df.iloc[-1]["probability_at_least_one"]
brier_score(predicted_probability=example_probability, actual_outcome=1)

In [ ]:
cleaned_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "nyc_crashes_clean.csv", dtype={"zip_code": str})
train_df, test_df = simple_train_test_split_by_date(cleaned_df, test_fraction=0.2)
len(train_df), len(test_df), train_df["date"].min(), test_df["date"].min()

In [ ]:
# Empirical daily counts vs Poisson(λ̂) for one bucket (same row as the rate table above)
poisson_fig = PROJECT_ROOT / "outputs" / "figures" / "empirical_vs_poisson_daily_counts.png"
out = plot_empirical_poisson_check(
    cleaned_df,
    zip_code=str(row["zip_code"]),
    day_of_week=str(row["day_of_week"]),
    hour=int(row["hour"]),
    output_path=poisson_fig,
)
out["sample_mean"], out["variance_to_mean"], out["n_days"], out["output_path"]